In [ ]:
# import the required libraries
import pandas as pd
from transformers import GPT2Tokenizer, GPT2Model
import torch
import numpy as np

# read the input data (cleaned embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title dataset is the dataset 
formed by merging & cleaning the embedded keywords and embedded title dataset.
"""
movies_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/gpt2_chunk_2.csv")

In [ ]:
movies_df.head(10)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,125795,Drama,tt0288092,sl,Kruh in mleko,Ivan is released a day early from a treatment ...,1.400,3/31/2001,Slovenian,6.200,8.0,125795.0,alcoholism,[8],-1.39311433e-01 9.86292958e-04 -4.82136905e-0...,-1.81071624e-01 -2.36139223e-01 -4.16063424e-0...
1,False,125796,Music,NaN,fr,Pascal Obispo - Live 98,NaN,0.600,5/4/1998,French,8.000,1.0,125796.0,concert,[18],-2.41738617e-01 -1.25076287e-04 -3.48370731e-0...,-1.90179907e-02 -3.48646283e-01 -1.88079491e-0...
2,False,125803,"Drama, Romance",tt0276015,ja,ã‚¯ãƒ­ã‚¨,Kotaro and Kuroe marry almost immediately in a...,1.272,6/15/2001,Japanese,6.000,2.0,125803.0,romance,"[8, 13]",-9.61342603e-02 -2.63273679e-02 -1.83916867e-0...,3.68895046e-02 -1.83421656e-01 1.22400271e-0...
3,False,125815,Drama,tt0285166,fr,H Story,An autobiographical docudrama about an attempt...,1.597,10/17/2001,"Japanese, French",5.800,6.0,125815.0,filmmaking,[8],6.05435707e-02 3.54522504e-02 -1.99616328e-0...,-2.05218315e-01 -1.37823150e-02 -1.91251829e-0...
4,False,125842,"Drama, TV Movie",tt0294918,en,Shot in the Heart,The story of a man coming to terms with the si...,1.968,10/13/2001,English,5.900,15.0,125842.0,woman director,"[8, 14]",1.09326810e-01 1.74034685e-02 -4.86597776e-0...,-9.57084447e-02 1.75823569e-01 2.35325590e-0...
5,False,125844,"Comedy, Drama",tt0256666,fr,Ce vieux rÃªve qui bouge,A chronicle of queer lust set against a greasy...,1.704,3/31/2001,French,6.154,14.0,125844.0,factory southern france rural area sexual attr...,"[5, 8]",2.19445139e-01 1.69175267e-02 -5.85175455e-0...,-3.35138575e-05 -6.00597501e-01 4.65537876e-0...
6,False,125849,"Adventure, Crime, Drama",tt0046979,en,Malaga,"A female former OSS agent is sent to Tangiers,...",2.730,6/24/1954,English,4.200,4.0,125849.0,smuggling (contraband) africa,"[17, 1, 8]",-1.45591617e-01 -1.84737504e-01 -5.89774013e-0...,3.08620334e-02 -2.40081921e-01 -1.66313499e-0...
7,False,125858,Drama,tt0388474,ja,Tokyo.Sora,Six women attempt to make it in current day To...,1.863,10/26/2002,Japanese,7.100,13.0,125858.0,female friendship,[8],2.55674124e-01 3.80863994e-01 -6.37534380e-0...,-6.92772586e-03 -3.74452651e-01 -1.11502007e-0...
8,False,125873,"Action, Adventure",tt0029951,en,Bulldog Drummond in Africa,Drummond has to leave for Morocco on his weddi...,2.398,8/5/1938,English,5.500,12.0,125873.0,africa secret agent bulldog drummond,"[19, 17]",-1.04584619e-01 5.31440526e-02 -3.64251852e-0...,1.02283619e-01 -2.83494860e-01 -4.57347929e-0...
9,False,125874,"Romance, Drama",tt1213921,en,Love & Savagery,"In 1969, a visiting geologist from Newfoundlan...",4.228,8/6/2009,English,4.857,7.0,125874.0,canada island nun religion ireland geologist c...,"[13, 8]",1.89914294e-02 -6.50355443e-02 -6.35266721e-0...,-3.19422930e-01 -1.54508457e-01 2.79957175e-0...


In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_df.columns

Index(['adult', 'id', 'genres', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
drop_columns = ['adult', 'genres', 'original_language',
       'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview
0,125795,tt0288092,Kruh in mleko,Ivan is released a day early from a treatment ...
1,125796,NaN,Pascal Obispo - Live 98,NaN
2,125803,tt0276015,ã‚¯ãƒ­ã‚¨,Kotaro and Kuroe marry almost immediately in a...
3,125815,tt0285166,H Story,An autobiographical docudrama about an attempt...
4,125842,tt0294918,Shot in the Heart,The story of a man coming to terms with the si...


In [ ]:
# reduce the size of the overview column to 500 characters of string
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column '{}': {}".format('overview', max_length))
movies_df['overview'] = movies_df['overview'].str.slice(0, 500)
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column after strip '{}': {}".format('overview', max_length))

Maximum length of column 'overview': 1172.0
Maximum length of column after strip 'overview': 500.0


In [ ]:
# load the pre-trained gpt-2 tokenizer and model
model_name='gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2Model.from_pretrained(model_name)

def create_word_embeddings(sentence):
    if pd.isna(sentence):
      return np.zeros(768)
    else:
      input_ids = tokenizer.encode(sentence, return_tensors='pt')
      with torch.no_grad():
          output = model(input_ids)
      embeddings = output.last_hidden_state.mean(dim=1).squeeze().numpy()
      return embeddings

# Apply the function to each row and create a new column 'overview_embedding'
movies_df['overview_embedding'] = movies_df['overview'].apply(create_word_embeddings)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [ ]:
movies_df['overview_embedding'] = movies_df['overview_embedding'].apply(lambda x: str(x))
movies_df.loc[0, 'overview_embedding']

'[-5.26635312e-02 -1.67690113e-01 -4.18948650e-01 -1.70245126e-01\n  3.91434915e-02 -2.79209673e-01  3.38688493e+00 -2.83075008e-03\n -2.82787055e-01  1.16119631e-01  7.94052482e-02  1.30446509e-01\n -1.24867544e-01 -1.04784288e-01  2.17577487e-01  1.40434131e-01\n -2.17979521e-01  2.30767727e-02 -3.15618254e-02 -6.88465118e-01\n  5.66072622e-03 -6.26846310e-03 -1.69816047e-01  2.03135043e-01\n  6.37637079e-03  4.93945144e-02 -3.40730816e-01  1.71334110e-02\n -5.97980544e-02  1.46474808e-01 -1.86298132e-01 -1.47615284e-01\n -1.89258471e-01 -1.00084409e-01  2.25733429e-01  4.17949855e-01\n  7.74551926e+01  1.01999246e-01 -9.57245752e-02  1.21751241e-01\n -1.08009890e-01  1.47235468e-01  5.69322743e-02 -3.00111145e-01\n -2.98158735e-01  1.52483927e-02  9.66040641e-02  5.89839695e-03\n  1.11099537e-02  1.27500251e-01 -5.53266285e-03  3.62226889e-02\n -4.28200737e-02 -1.70204118e-01  1.64137706e-01  1.55048764e+00\n  1.29525587e-01 -1.07755244e-01  2.41793364e-01  2.24956080e-01\n -1.32091

In [ ]:
# convert the embedded column array to embedded string and clean the overview_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_df['overview_embedding_string'] = movies_df['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '').str.replace(',', '')

In [ ]:
movies_df.loc[0, 'overview_embedding_string']

'-5.26635312e-02 -1.67690113e-01 -4.18948650e-01 -1.70245126e-01  3.91434915e-02 -2.79209673e-01  3.38688493e+00 -2.83075008e-03 -2.82787055e-01  1.16119631e-01  7.94052482e-02  1.30446509e-01 -1.24867544e-01 -1.04784288e-01  2.17577487e-01  1.40434131e-01 -2.17979521e-01  2.30767727e-02 -3.15618254e-02 -6.88465118e-01  5.66072622e-03 -6.26846310e-03 -1.69816047e-01  2.03135043e-01  6.37637079e-03  4.93945144e-02 -3.40730816e-01  1.71334110e-02 -5.97980544e-02  1.46474808e-01 -1.86298132e-01 -1.47615284e-01 -1.89258471e-01 -1.00084409e-01  2.25733429e-01  4.17949855e-01  7.74551926e+01  1.01999246e-01 -9.57245752e-02  1.21751241e-01 -1.08009890e-01  1.47235468e-01  5.69322743e-02 -3.00111145e-01 -2.98158735e-01  1.52483927e-02  9.66040641e-02  5.89839695e-03  1.11099537e-02  1.27500251e-01 -5.53266285e-03  3.62226889e-02 -4.28200737e-02 -1.70204118e-01  1.64137706e-01  1.55048764e+00  1.29525587e-01 -1.07755244e-01  2.41793364e-01  2.24956080e-01 -1.32091999e-01 -1.25333458e-01 -8.4363

In [ ]:
drop_columns = ['overview_embedding']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview,overview_embedding_string
0,125795,tt0288092,Kruh in mleko,Ivan is released a day early from a treatment ...,-5.26635312e-02 -1.67690113e-01 -4.18948650e-0...
1,125796,NaN,Pascal Obispo - Live 98,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...
2,125803,tt0276015,ã‚¯ãƒ­ã‚¨,Kotaro and Kuroe marry almost immediately in a...,2.42314652e-01 7.91362002e-02 -5.16620994e-0...
3,125815,tt0285166,H Story,An autobiographical docudrama about an attempt...,2.89413005e-01 1.82686314e-01 -4.86765146e-0...
4,125842,tt0294918,Shot in the Heart,The story of a man coming to terms with the si...,-1.55527443e-01 -2.72100233e-02 -7.00956941e-0...


In [ ]:
movies_df['overview_embedding_string'] = movies_df['overview_embedding_string'].str.strip()

In [ ]:
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/old_final_mkg_dataset_gpt2.csv")

In [ ]:
merged_df = pd.merge(movies_df, movies_keywords_genres, on=['id', 'imdb_id', 'original_title'], how='inner')

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,125795,tt0288092,Kruh in mleko,Ivan is released a day early from a treatment ...,-5.26635312e-02 -1.67690113e-01 -4.18948650e-0...,False,Drama,sl,Ivan is released a day early from a treatment ...,1.400,3/31/2001,Slovenian,6.2,8.0,125795.0,alcoholism,[8],-1.39311433e-01 9.86292958e-04 -4.82136905e-0...,-1.81071624e-01 -2.36139223e-01 -4.16063424e-0...
1,125796,NaN,Pascal Obispo - Live 98,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Music,fr,NaN,0.600,5/4/1998,French,8.0,1.0,125796.0,concert,[18],-2.41738617e-01 -1.25076287e-04 -3.48370731e-0...,-1.90179907e-02 -3.48646283e-01 -1.88079491e-0...
2,125803,tt0276015,ã‚¯ãƒ­ã‚¨,Kotaro and Kuroe marry almost immediately in a...,2.42314652e-01 7.91362002e-02 -5.16620994e-01...,False,"Drama, Romance",ja,Kotaro and Kuroe marry almost immediately in a...,1.272,6/15/2001,Japanese,6.0,2.0,125803.0,romance,"[8, 13]",-9.61342603e-02 -2.63273679e-02 -1.83916867e-0...,3.68895046e-02 -1.83421656e-01 1.22400271e-0...
3,125815,tt0285166,H Story,An autobiographical docudrama about an attempt...,2.89413005e-01 1.82686314e-01 -4.86765146e-01...,False,Drama,fr,An autobiographical docudrama about an attempt...,1.597,10/17/2001,"Japanese, French",5.8,6.0,125815.0,filmmaking,[8],6.05435707e-02 3.54522504e-02 -1.99616328e-0...,-2.05218315e-01 -1.37823150e-02 -1.91251829e-0...
4,125842,tt0294918,Shot in the Heart,The story of a man coming to terms with the si...,-1.55527443e-01 -2.72100233e-02 -7.00956941e-0...,False,"Drama, TV Movie",en,The story of a man coming to terms with the si...,1.968,10/13/2001,English,5.9,15.0,125842.0,woman director,"[8, 14]",1.09326810e-01 1.74034685e-02 -4.86597776e-0...,-9.57084447e-02 1.75823569e-01 2.35325590e-0...


In [ ]:
merged_df.shape[0]

46946

In [ ]:
merged_df.columns

Index(['id', 'imdb_id', 'original_title', 'overview_x',
       'overview_embedding_string', 'adult', 'genres', 'original_language',
       'overview_y', 'popularity', 'release_date', 'spoken_languages',
       'vote_average', 'vote_count', 'movie_id', 'keywords', 'genres_indices',
       'keywords_embedding_string', 'original_title_embedding_string'],
      dtype='object')

In [ ]:
nan_count = merged_df['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding' has {nan_count} NaN values.")
nan_count = merged_df['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
merged_df.dropna(subset=['overview_embedding_string'], inplace=True)
merged_df.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
merged_df.shape[0]

46946

In [ ]:
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/final_mkg_dataset_gpt2_chunk_2.csv', index=False)

In [ ]:
#  Test

In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,125795,tt0288092,Kruh in mleko,Ivan is released a day early from a treatment ...,-5.26635312e-02 -1.67690113e-01 -4.18948650e-0...,False,Drama,sl,Ivan is released a day early from a treatment ...,1.400,3/31/2001,Slovenian,6.2,8.0,125795.0,alcoholism,[8],-1.39311433e-01 9.86292958e-04 -4.82136905e-0...,-1.81071624e-01 -2.36139223e-01 -4.16063424e-0...
1,125796,NaN,Pascal Obispo - Live 98,NaN,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,False,Music,fr,NaN,0.600,5/4/1998,French,8.0,1.0,125796.0,concert,[18],-2.41738617e-01 -1.25076287e-04 -3.48370731e-0...,-1.90179907e-02 -3.48646283e-01 -1.88079491e-0...
2,125803,tt0276015,ã‚¯ãƒ­ã‚¨,Kotaro and Kuroe marry almost immediately in a...,2.42314652e-01 7.91362002e-02 -5.16620994e-01...,False,"Drama, Romance",ja,Kotaro and Kuroe marry almost immediately in a...,1.272,6/15/2001,Japanese,6.0,2.0,125803.0,romance,"[8, 13]",-9.61342603e-02 -2.63273679e-02 -1.83916867e-0...,3.68895046e-02 -1.83421656e-01 1.22400271e-0...
3,125815,tt0285166,H Story,An autobiographical docudrama about an attempt...,2.89413005e-01 1.82686314e-01 -4.86765146e-01...,False,Drama,fr,An autobiographical docudrama about an attempt...,1.597,10/17/2001,"Japanese, French",5.8,6.0,125815.0,filmmaking,[8],6.05435707e-02 3.54522504e-02 -1.99616328e-0...,-2.05218315e-01 -1.37823150e-02 -1.91251829e-0...
4,125842,tt0294918,Shot in the Heart,The story of a man coming to terms with the si...,-1.55527443e-01 -2.72100233e-02 -7.00956941e-0...,False,"Drama, TV Movie",en,The story of a man coming to terms with the si...,1.968,10/13/2001,English,5.9,15.0,125842.0,woman director,"[8, 14]",1.09326810e-01 1.74034685e-02 -4.86597776e-0...,-9.57084447e-02 1.75823569e-01 2.35325590e-0...


In [ ]:
# merge the embedded keywords, title and overview columns to form the input gpt2 embedded dataset chunk-2
drop_columns = ['imdb_id', 'original_title', 'overview_x', 'adult', 'original_language', 'overview_y', 'popularity', 'spoken_languages', 'vote_average', 'vote_count', 'movie_id', 'keywords']
merged_df = merged_df.drop(columns=drop_columns)
merged_df.head(5)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,125795,-5.26635312e-02 -1.67690113e-01 -4.18948650e-0...,Drama,3/31/2001,[8],-1.39311433e-01 9.86292958e-04 -4.82136905e-0...,-1.81071624e-01 -2.36139223e-01 -4.16063424e-0...
1,125796,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Music,5/4/1998,[18],-2.41738617e-01 -1.25076287e-04 -3.48370731e-0...,-1.90179907e-02 -3.48646283e-01 -1.88079491e-0...
2,125803,2.42314652e-01 7.91362002e-02 -5.16620994e-01...,"Drama, Romance",6/15/2001,"[8, 13]",-9.61342603e-02 -2.63273679e-02 -1.83916867e-0...,3.68895046e-02 -1.83421656e-01 1.22400271e-0...
3,125815,2.89413005e-01 1.82686314e-01 -4.86765146e-01...,Drama,10/17/2001,[8],6.05435707e-02 3.54522504e-02 -1.99616328e-0...,-2.05218315e-01 -1.37823150e-02 -1.91251829e-0...
4,125842,-1.55527443e-01 -2.72100233e-02 -7.00956941e-0...,"Drama, TV Movie",10/13/2001,"[8, 14]",1.09326810e-01 1.74034685e-02 -4.86597776e-0...,-9.57084447e-02 1.75823569e-01 2.35325590e-0...


In [ ]:
merged_df.shape[0]

46946

In [ ]:
# export the embedded keywords, title and overview dataset (chunk-2)
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_2.csv', index=False)

In [ ]:
# Testing read

In [ ]:
test_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_2.csv")

In [ ]:
test_df.shape[0]

46946

In [ ]:
test_df.columns

Index(['id', 'overview_embedding_string', 'genres', 'release_date',
       'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
test_df.head(4)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,125795,-5.26635312e-02 -1.67690113e-01 -4.18948650e-0...,Drama,3/31/2001,[8],-1.39311433e-01 9.86292958e-04 -4.82136905e-0...,-1.81071624e-01 -2.36139223e-01 -4.16063424e-0...
1,125796,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,Music,5/4/1998,[18],-2.41738617e-01 -1.25076287e-04 -3.48370731e-0...,-1.90179907e-02 -3.48646283e-01 -1.88079491e-0...
2,125803,2.42314652e-01 7.91362002e-02 -5.16620994e-01...,"Drama, Romance",6/15/2001,"[8, 13]",-9.61342603e-02 -2.63273679e-02 -1.83916867e-0...,3.68895046e-02 -1.83421656e-01 1.22400271e-0...
3,125815,2.89413005e-01 1.82686314e-01 -4.86765146e-01...,Drama,10/17/2001,[8],6.05435707e-02 3.54522504e-02 -1.99616328e-0...,-2.05218315e-01 -1.37823150e-02 -1.91251829e-0...
